In [1]:
# 셀 1 — GPU 확인 + 설치
!nvidia-smi -L                      # "Tesla T4"가 보여야 함
!pip -q install FlagEmbedding

GPU 0: NVIDIA L4 (UUID: GPU-308b0385-5541-4126-8e9b-3b99357d4d77)


In [2]:
# 셀 2(수정) — VM이 저장소에서 입력 직접 확보
!wget -q https://raw.githubusercontent.com/worldoftoddl/auditPaper_assist/main/index/embed_input.jsonl -O embed_input.jsonl
n = sum(1 for _ in open("embed_input.jsonl", encoding="utf-8"))
assert n == 10063, f"행 수 불일치: {n}"
print("입력 확보 완료:", n, "행")

입력 확보 완료: 10063 행


In [3]:
# 셀 3 — 인코딩 (T4 기준 5~10분)
import json, numpy as np
rows  = [json.loads(l) for l in open('embed_input.jsonl', encoding='utf-8')]
cids  = [r['cid']  for r in rows]
texts = [r['text'] for r in rows]
assert len(cids) == len(set(cids)) == 10063, len(cids)

from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
out = model.encode(texts, batch_size=64, max_length=8192,
                   return_dense=True, return_sparse=False,
                   return_colbert_vecs=False)
emb = np.asarray(out['dense_vecs'], dtype='float32')
assert emb.shape == (10063, 1024), emb.shape

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 158/158 [00:34<00:00,  4.52it/s]


In [4]:
# 정규화 검증 — 매니페스트 계약(normalize=True) 준수 보증
n = np.linalg.norm(emb, axis=1)
if not np.allclose(n, 1.0, atol=1e-3):
    emb = emb / n[:, None]
print("norm range:", n.min(), n.max())

norm range: 0.99967366 1.0003407


In [6]:
# 셀 4 — 저장·다운로드 (약 41MB)
np.save('embeddings.npy', emb)
json.dump(cids, open('cids.json', 'w'), ensure_ascii=False)
import hashlib
print("sha256:", hashlib.sha256(open('embeddings.npy','rb').read()).hexdigest())
!curl --upload-file embeddings.npy 

sha256: 75b967e2e0b9587d5a363b78016f86aa5ff8e103e23b8c2e642d212b364e0a26
curl: no URL specified!
curl: try 'curl --help' or 'curl --manual' for more information


In [14]:
from getpass import getpass
qurl = getpass("Qdrant 클러스터 URL: ").rstrip("/")
!curl -sI -m 10 {qurl} | head -1
!curl -sI -m 10 https://codeload.github.com | head -1

HTTP/2 403 
HTTP/2 301 


In [15]:
import os
from getpass import getpass
os.environ["QDRANT_URL"] = qurl
os.environ["QDRANT_API_KEY"] = getpass("QDRANT_API_KEY: ")

!wget -q https://codeload.github.com/worldoftoddl/auditPaper_assist/tar.gz/refs/heads/main -O repo.tgz
!mkdir -p repo && tar xzf repo.tgz -C repo --strip-components=1
assert os.path.isfile("repo/scripts/build_index.py"), "저장소 확보 실패 — 중단"
!pip -q install qdrant-client kiwipiepy
!cp /content/embeddings.npy /content/cids.json repo/index/
%cd repo
!python scripts/build_index.py --stage upsert --embeddings index/embeddings.npy --cids index/cids.json

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 29.5 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 139.1 MB/s eta 0:00:0000:010:01
/content/repo
[F] 용어 사전 파생...
[E] sparse 벡터 (kiwipiepy + BM25 문서측 가중)...
  sparse 토큰화 2000/10063
  sparse 토큰화 4000/10063
  sparse 토큰화 6000/10063
  sparse 토큰화 8000/10063
  sparse 토큰화 10000/10063
[G] Qdrant 적재: standards_20250829_bgem3
  업서트 256/10063
  업서트 512/10063
  업서트 768/10063
  업서트 1024/10063
  업서트 1280/10063
  업서트 1536/10063
  업서트 1792/10063
  업서트 2048/10063
  업서트 2304/10063
  업서트 2560/10063
  업서트 2816/10063
  업서트 3072/10063
  업서트 3328/10063
  업서트 3584/10063
  업서트 3840/10063
  업서트 4096/10063
  업서트 4352/10063
  업서트 4608/10063
  업서트 4864/10063
  업서트 5120/10063
  업서트 5376/10063
  업서트 5632/10063
  업서트 5888/10063
  업서트 6144/10063
  업서트 6400/10063
  업서트 6656/10063
  업서트 6912/10063
  업서트 71

In [16]:
for f in ["index/manifest.json", "index/vocab.json", "index/glossary.jsonl"]:
    print(f"===== {f} ====="); print(open(f, encoding="utf-8").read())

===== index/manifest.json =====
{
  "corpus_commit": "",
  "collection": "standards_20250829_bgem3",
  "uuid5_namespace": "github.com/worldoftoddl/auditPaper_assist",
  "embedding": {
    "model": "BAAI/bge-m3",
    "revision": "main",
    "normalize_embeddings": true,
    "dim": 1024,
    "max_tokens": 8192,
    "library": "sentence-transformers",
    "embeddings_sha256": "75b967e2e0b9587d5a363b78016f86aa5ff8e103e23b8c2e642d212b364e0a26",
    "embedding_device": "unknown",
    "embedding_runtime": "unknown"
  },
  "sparse": {
    "tokenizer": "kiwipiepy",
    "kiwipiepy_version": "0.23.2",
    "keep_tags": [
      "NNG",
      "NNP",
      "SH",
      "SL",
      "SN",
      "XR"
    ],
    "lowercase_SL": true,
    "k1": 1.5,
    "b": 0.75,
    "avgdl": 69.2988,
    "vocab_size": 4905,
    "doc_count": 10063,
    "query_side": "토큰당 1.0 (IDF는 Qdrant 서버가 적용)"
  },
  "paragraphs": 10062,
  "points": 10063,
  "para_type_expected": {
    "정의": 280,
    "참조": 14
  },
  "split": {
    "KSA: